# SRAG — Brazilian Severe Acute Respiratory Syndrome Surveillance

This notebook demonstrates the **`srag`** accessor for the official
Brazilian national SRAG dataset (Síndrome Respiratória Aguda Grave),
integrating **Sivep-Gripe** (severe/hospitalised cases) and
**e-SUS Notifica** (COVID-19 notifications), published per year on the
[OpenData SUS portal](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026).

Key characteristics:

- **Line-level records** — demographics, symptoms, comorbidities, ICU
  admission, ventilation, laboratory panels (RT-PCR/antigen for
  SARS-CoV-2, influenza A/B, VSR, ...) and outcomes;
- All 5,570 Brazilian municipalities (IBGE geocodes), 2019–present;
- Annual files 0.5–3 GB: past years are frozen snapshots, the current
  year is a "banco vivo" refreshed weekly.

> **Note** — cells that download annual case files (hundreds of MB to
> GB) and the weekly-count aggregation over them are shown but not
> executed in this copy; everything else (portal discovery, IBGE
> helpers, validation, registry) runs live.

In [1]:
from epidatasets.sources.srag_opendata_sus import (
    SRAGOpenDataSUSAccessor,
    SRAGDataError,
)

srag = SRAGOpenDataSUSAccessor()  # optional: live_ttl_hours=24
print(srag)

Brazilian national SRAG surveillance (Sivep-Gripe / e-SUS Notifica): line-level severe acute respiratory syndrome cases with demographic, clinical, laboratory and outcome variables, published per year on OpenData SUS (weekly-updated 'banco vivo' for the current year). (https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026)


## 1. Available years and resources

In [2]:
years = srag.available_years()
years

[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [3]:
resources = srag.list_resources()
resources[resources["format"] == "CSV"][["year", "name", "url"]]

,year,name,url
0,2019,2019- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
3,2020,2020- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
6,2021,2021- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
9,2022,2022- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
12,2023,2023- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
15,2024,2024- Banco vivo 23/03/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
18,2025,2025- Banco vivo 28/09/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....
23,2026,2026- Banco vivo 28/09/2026 - CSV,https://s3.sa-east-1.amazonaws.com/ckan.saude....


## 2. Loading one year with composable filters

`get_year` streams the file, reads it in chunks and applies filters
chunk-by-chunk, so a full year never has to fit in memory. Pass
`columns=[...]` to narrow the output early.

In [ ]:
# severe COVID-19 deaths among 60+ residents of Rio de Janeiro municipality
rj_covid_deaths = srag.get_year(
    2024,
    state="RJ",
    municipality_code=3304557,     # IBGE code (6 or 7 digits)
    pathogen="SARS-CoV-2",
    outcome="death",
    age_min=60,
    columns=["DT_NOTIFIC", "SEM_NOT", "DT_SIN_PRI", "CS_SEXO",
             "NU_IDADE_N", "DT_EVOLUCA"],
)
rj_covid_deaths.head()

In [ ]:
# other pathogen filters: "Influenza", "Influenza A", "Influenza B",
# "VSR"/"RSV", "Adenovirus", "Parainfluenza", "Metapneumovirus", "Rhinovirus"
flu = srag.get_year(2023, state="SP", pathogen="Influenza", week_start=20, week_end=30)
len(flu)

In [ ]:
# lazy chunk iteration for very large years (e.g. 2020-2022 pandemic peaks)
total = 0
for chunk in srag.get_year(2021, chunksize=500_000,
                           columns=["SEM_NOT", "SG_UF", "EVOLUCAO"]):
    total += len(chunk)
print(f"{total:,} records")

## 3. Weekly counts by geography and pathogen

In [ ]:
ts = srag.get_weekly_counts(
    years=[2022, 2023, 2024],
    groupby="state",                       # or "municipality", "region", "macroregion"
    pathogens=["SARS-CoV-2", "Influenza", "VSR"],
)
ts.head()

In [ ]:
import matplotlib.pyplot as plt

subset = ts[ts["SG_UF"].isin(["SP", "RJ", "AM"])]
fig, ax = plt.subplots(figsize=(12, 4))
for (uf), grp in subset.groupby("SG_UF"):
    (grp["year"] + grp["epi_week"] / 53).plot.line(
        ax=ax, label=None, color="gray", alpha=0.3, legend=False
    )
ts_brazil = ts.groupby(["year", "epi_week"])["total"].sum().reset_index()
ax.plot(ts_brazil["year"] + ts_brazil["epi_week"] / 53, ts_brazil["total"],
        color="crimson", label="Brazil total")
ax.set_ylabel("weekly SRAG notifications"); ax.legend();

## 4. IBGE municipality helpers

Filters accept IBGE geocodes; the accessor resolves them to/from names
via the IBGE `localidades` API (cached ~30 days).

In [4]:
rios = srag.get_municipalities(state="RJ")
print(f"{len(rios)} municipalities in RJ")
rios.head()

92 municipalities in RJ


,municipality_code,municipality_name,state,state_name
0,3300100,Angra dos Reis,RJ,Rio de Janeiro
1,3300159,Aperibé,RJ,Rio de Janeiro
2,3300209,Araruama,RJ,Rio de Janeiro
3,3300225,Areal,RJ,Rio de Janeiro
4,3300233,Armação dos Búzios,RJ,Rio de Janeiro


In [5]:
srag.municipality_name(3304557)

'Rio de Janeiro'

In [6]:
srag.municipality_code("São Paulo", state="SP")

'3550308'

## 5. Validation and error handling

In [7]:
try:
    srag.get_year(2031)                     # year not on the portal
except ValueError as e:
    print("ValueError:", str(e)[:120], "...")

ValueError: No CSV resource for SRAG year 2031. Available years: 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 ...


In [8]:
try:
    srag.get_year(2024, pathogen="Ebola")   # unsupported pathogen
except ValueError as e:
    print("ValueError:", e)

ValueError: Unknown pathogen 'Ebola'. Supported: adenovirus, covid-19, covid19, influenza, influenza a, influenza b, metapneumovirus, parainfluenza, rhinovirus, rsv, sars-cov-2, sars_cov_2, vsr


In [ ]:
try:
    srag.get_year(2024, state="RJ")         # (download not run here)
except SRAGDataError as e:
    print("SRAGDataError:", str(e)[:120], "...")

## 6. Using the plugin registry

In [9]:
from epidatasets import get_source

acc = get_source("srag")
acc.list_countries()

,country_code,country_name
0,BR,Brazil


## References

- Dataset page: https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026
- Reference scripts (R): https://gitlab.com/cgcovid/dados-abertos
- Variable dictionary (`dicionário de dados`) is published alongside the data
- Complementary access: `epidatasets.sources.opendatasus` (catalog) and
  `epidatasets.sources.datasus_pysus` (legacy DATASUS route)